In [3]:
import sqlite3
import pandas as pd
import seaborn as sns


notebooks/
├── 01_eda.ipynb              # Exploration (notebook)
├── 02_woe_iv.ipynb           # Analyse WOE/IV (notebook qui importe src/woe.py)
├── 03_scorecard.ipynb        # Analyse scorecard (notebook qui importe src/scorecard.py)
├── 04_xgboost.ipynb          # Analyse XGBoost (notebook qui importe src/challenger.py)
├── 05_calibration.ipynb      # Analyse calibration (notebook qui importe src/calibration.py)
└── 06_backtesting.ipynb      # Analyse back-testing (notebook qui importe src/backtesting.py)

src/
├── data_loader.py             # Code de production (testable)
├── woe.py                     # Code de production (testable)
├── scorecard.py               # Code de production (testable)
├── calibration.py             # Code de production (testable)
├── challenger.py              # Code de production (testable)
├── backtesting.py             # Code de production (testable)
└── utils.py                   # Fonctions utilitaires

tests/
├── test_data_loader.py        # Tests
├── test_woe.py                # Tests
├── test_scorecard.py          # Tests
├── test_calibration.py        # Tests
├── test_challenger.py         # Tests
└── test_backtesting.py        # Tests

In [27]:
# connection a la bdd et creation deu df
 
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sqlalchemy import create_engine

# Option A : lire depuis le fichier SQLite local via SQLAlchemy
engine = create_engine("sqlite:///../data/processed/german_credit.db")
df = pd.read_sql_query("SELECT * FROM clients", engine)
engine.dispose()  # libère les connexions ouvertes


Separtaiton d datatset et initialisations 

In [52]:
from sklearn.model_selection import train_test_split


target_col = "res"
feature_cols = [c for c in df.columns if c != target_col]

X = df[feature_cols]
y = df[target_col]



X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [53]:
type(X_train)

pandas.DataFrame

In [38]:
df

,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,res
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,0
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,1
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,0
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,0
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,A14,12,A32,A42,1736,A61,A74,3,A92,A101,...,A121,31,A143,A152,1,A172,1,A191,A201,0
996,A11,30,A32,A41,3857,A61,A73,4,A91,A101,...,A122,40,A143,A152,1,A174,1,A192,A201,0
997,A14,12,A32,A43,804,A61,A75,4,A93,A101,...,A123,38,A143,A152,1,A173,1,A191,A201,0
998,A11,45,A32,A43,1845,A61,A73,4,A93,A101,...,A124,23,A143,A153,1,A173,1,A192,A201,1


### Partie 6: Binning pooptimal et Weaight of Evidence (WoE)


In [ ]:
def binning(df,column):
 

In [55]:
X_train

,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status_sex,other_debtors,residence_since,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker
10,A12,12,A32,A40,1295,A61,A72,3,A92,A101,1,A123,25,A143,A151,1,A173,1,A191,A201
82,A14,18,A32,A49,1568,A62,A73,3,A92,A101,4,A122,24,A143,A151,1,A172,1,A191,A201
827,A14,18,A30,A49,4165,A61,A73,2,A93,A101,2,A123,36,A142,A152,2,A173,2,A191,A201
410,A12,24,A32,A43,1967,A61,A75,4,A92,A101,4,A123,20,A143,A152,1,A173,1,A192,A201
48,A14,11,A34,A40,7228,A61,A73,1,A93,A101,4,A122,39,A143,A152,2,A172,1,A191,A201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,A14,36,A32,A42,3349,A61,A73,4,A92,A101,2,A123,28,A143,A152,1,A174,1,A192,A201
759,A11,12,A34,A40,691,A61,A75,4,A93,A101,3,A122,35,A143,A152,2,A173,1,A191,A201
298,A14,18,A32,A42,2515,A61,A73,3,A93,A101,4,A121,43,A143,A152,1,A173,1,A192,A201
417,A11,18,A33,A46,8471,A65,A73,1,A92,A101,2,A123,23,A143,A151,2,A173,1,A192,A201


In [79]:
from optbinning import BinningProcess, OptimalBinning


cat_cols = (
    X_train[feature_cols]
    .select_dtypes(include=["object", "string", "str"])
    .columns.to_list()  
)
    

    

# 1. Binning d'une variable numérique
optb = OptimalBinning(
    name="age",
    dtype="numerical",
    solver="cp",
    monotonic_trend="auto",
    min_bin_size=0.05,
)
optb.fit(X_train["age"], y_train)
age_table = optb.binning_table.build()
print(age_table)
print("IV total (age) :", age_table.loc["Totals", "IV"])

# 2. Binning catégoriel
optb_cat = OptimalBinning(name="purpose", dtype="categorical", solver="cp")
optb_cat.fit(X_train["purpose"], y_train)
cat_table = optb_cat.binning_table.build()
print(cat_table)
print("IV total (purpose) :", cat_table.loc["Totals", "IV"])

# 3. Pipeline multi-variables (WoE)
def Multi_Binning():

    binning_process = BinningProcess(
        variable_names=feature_cols,
        categorical_variables=cat_cols,
    )
    binning_process.fit(X_train, y_train)
    df_woe_train = binning_process.transform(X_train, metric="woe")
    df_woe_test=binning_process.transform(X_test,metric="woe")
    df_woe_val=binning_process.transform(X_val,metric="woe")

    # Résumé global des IV par variable
    print(
        binning_process.summary()[["name", "dtype", "status", "iv"]].sort_values(
            "iv", ascending=False
        )       
    )
    return df_woe_train,df_woe_test,df_woe_val,binning_process

                   Bin  Count  Count (%)  Non-event  Event  Event rate  \
0        (-inf, 22.50)     39   0.055714         26     13    0.333333   
1       [22.50, 25.50)     85   0.121429         48     37    0.435294   
2       [25.50, 29.50)    123   0.175714         83     40    0.325203   
3       [29.50, 34.50)    138   0.197143         96     42    0.304348   
4         [34.50, inf)    315   0.450000        237     78    0.247619   
5              Special      0   0.000000          0      0    0.000000   
6              Missing      0   0.000000          0      0    0.000000   
Totals                    700   1.000000        490    210    0.300000   

             WoE        IV        JS  
0      -0.154151  0.001363  0.000170  
1      -0.587015  0.045923  0.005659  
2      -0.117337  0.002474  0.000309  
3      -0.020619  0.000084  0.000011  
4       0.264053  0.029639  0.003694  
5            0.0  0.000000  0.000000  
6            0.0  0.000000  0.000000  
Totals            0.0

pandas.Series

On remareque que l'inforation est conservee dans quelsues variables et que beacupiu n'apportent rien au modele.
On enleve donc : foreign_worker,num_independants,telephone

In [80]:
df_woe_val

NameError: name 'df_woe_val' is not defined

In [81]:
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
# df
df_woe_train,df_woe_test,df_woe_val,binning_process=Multi_Binning()
df_woe=df_woe_train
# 1. Filtrer les variables par IV (ex. IV >= 0.05 pour garder le signal utile)
summary = binning_process.summary()
#selection en prenant que les vars avec IV >= 0.05
selected_cols = summary.loc[summary["iv"] >= 0.05, "name"].tolist()
print(f"Variables retenues ({len(selected_cols)}) :", selected_cols)

# 2. Matrice de corrélation sur les variables transformées en WoE
df_woe_sel = df_woe[selected_cols]
corr_matrix = df_woe_sel.corr()

# 3. Repérer les paires redondantes (|corr| > 0.5)
threshold = 0.50
high_corr_pairs = []
cols = corr_matrix.columns
for i in range(len(cols)):
  for j in range(i + 1, len(cols)):
    c_val = corr_matrix.iloc[i, j]
    if abs(c_val) >= threshold:
      iv_i = summary.loc[summary["name"] == cols[i], "iv"].iloc[0]
      iv_j = summary.loc[summary["name"] == cols[j], "iv"].iloc[0]
      high_corr_pairs.append({
          "var_1": cols[i],
          "iv_1": round(iv_i, 4),
          "var_2": cols[j],
          "iv_2": round(iv_j, 4),
          "corr": round(c_val, 3),
      })

df_pairs = pd.DataFrame(high_corr_pairs)
df_pairs = pd.DataFrame(high_corr_pairs)

print("\n=== Paires corrélées (|r| >= 0.50) ===")
if df_pairs.empty:
  print("Aucune paire au-dessus du seuil.")
else:
  df_pairs = df_pairs.sort_values(by="corr", key=abs, ascending=False)
  print(df_pairs.to_string(index=False))

# 4. Calcul du VIF (Variance Inflation Factor) sur df_woe_sel
X_vif = df_woe_sel.assign(const=1.0)
vif_df = pd.DataFrame({
    "variable": df_woe_sel.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(len(df_woe_sel.columns))
    ],
}).sort_values("VIF", ascending=False)

print("\n=== VIF (redondance multivariable) ===")
print(vif_df.to_string(index=False))

                       name        dtype   status        iv
0           checking_status  categorical  OPTIMAL  0.638775
1                  duration    numerical  OPTIMAL  0.278819
4             credit_amount    numerical  OPTIMAL  0.265101
2            credit_history  categorical  OPTIMAL  0.254223
5                   savings  categorical  OPTIMAL  0.233467
6                employment  categorical  OPTIMAL  0.140286
11                 property  categorical  OPTIMAL  0.140196
3                   purpose  categorical  OPTIMAL  0.131766
13  other_installment_plans  categorical  OPTIMAL  0.116911
14                  housing  categorical  OPTIMAL   0.10204
12                      age    numerical  OPTIMAL  0.079483
8       personal_status_sex  categorical  OPTIMAL  0.039194
7          installment_rate    numerical  OPTIMAL  0.033767
16                      job  categorical  OPTIMAL  0.015589
18                telephone  categorical  OPTIMAL  0.002936
15         existing_credits    numerical

Commentaire : toute sles vatraibles ont un VIF informeir a 5 donc il n'y a pas de probleme de firte collineairte

Tests de satbilite

In [82]:
import numpy as np
import pandas as pd

def calc_psi(expected, actual, bins=10):
    """Calcule le PSI entre deux distributions."""
    # Binning sur les quantiles de expected
    breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
    breakpoints = np.unique(breakpoints)  # évite les doublons
    
    # Comptage par bin
    exp_counts = np.histogram(expected, bins=breakpoints)[0]
    act_counts = np.histogram(actual, bins=breakpoints)[0]
    
    # Pourcentages avec lissage (évite log(0))
    exp_pct = (exp_counts + 0.5) / (exp_counts.sum() + 0.5 * len(exp_counts))
    act_pct = (act_counts + 0.5) / (act_counts.sum() + 0.5 * len(act_counts))
    
    # PSI
    psi = np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))
    return psi

# Utilisation
psi_value = calc_psi(X_train["age"], X_test["age"])
print(f"PSI = {psi_value:.4f}")

# Interprétation
if psi_value < 0.1:
    print("Stable (pas de drift)")
elif psi_value < 0.25:
    print("Drift modéré (à surveiller)")
else:
    print("Drift significatif (recalibration nécessaire)")

PSI = 0.0820
Stable (pas de drift)


In [ ]:
df.drop

In [99]:
df_woe_test

,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status_sex,residence_since,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone
258,1.184134,0.251314,-0.082842,0.554501,2.043074,-0.098061,-0.579034,0.292136,-0.246088,0.036368,-0.034804,-0.587015,0.174622,0.214468,-0.035545,0.039556,0.005666,0.066521
662,1.184134,-0.176130,-0.082842,-0.111591,0.239754,-0.301787,0.194156,-0.149489,0.163420,-0.010050,0.471278,0.264053,0.174622,0.214468,0.060961,0.039556,0.005666,-0.044150
915,-0.415165,-0.987060,-1.090920,-0.111591,-0.965081,-0.301787,0.074691,0.292136,-0.246088,-0.010050,0.051449,-0.020619,-0.746202,0.214468,-0.035545,-0.273497,0.005666,0.066521
192,-0.415165,-0.176130,-0.082842,-0.241162,-0.311780,-0.301787,0.074691,-0.149489,0.163420,-0.010050,-0.034804,0.264053,0.174622,0.214468,-0.035545,0.039556,-0.029853,0.066521
570,-0.763916,-0.176130,-0.082842,-0.111591,0.251314,-0.301787,-0.579034,-0.149489,-0.246088,0.036368,0.471278,-0.587015,0.174622,-0.468861,-0.035545,0.104261,0.005666,0.066521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533,1.184134,-0.176130,-0.082842,0.384846,0.128712,-0.098061,0.517018,-0.149489,0.079464,-0.010050,0.051449,-0.117337,0.174622,0.214468,-0.035545,0.039556,0.005666,0.066521
696,-0.415165,0.251314,-0.082842,0.384846,0.128712,-0.301787,0.517018,-0.149489,0.163420,-0.010050,0.471278,-0.117337,0.174622,0.214468,0.060961,0.039556,0.005666,-0.044150
993,-0.763916,-0.474623,-0.082842,-0.111591,-0.311780,-0.301787,-0.684779,-0.149489,0.163420,-0.010050,0.051449,-0.020619,0.174622,0.214468,-0.035545,-0.273497,0.005666,0.066521
47,-0.763916,1.455287,-0.082842,0.554501,0.128712,0.733153,-0.684779,0.292136,-0.246088,-0.010050,0.051449,-0.587015,0.174622,-0.468861,-0.035545,-0.273497,0.005666,0.066521


In [102]:
df_woe_train2=df_woe_train.drop(columns=["other_debtors","foreign_worker"])
df_woe_test2=df_woe_test.drop(columns=["other_debtors","foreign_worker"])
df_woe_val2=df_woe_val.drop(columns=["other_debtors","foreign_worker"])




KeyError: "['other_debtors', 'foreign_worker'] not found in axis"

In [107]:
df_woe_val

,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status_sex,residence_since,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone
309,-0.415165,0.499776,-0.082842,-0.180353,-0.310993,-0.301787,0.074691,-0.149489,0.079464,0.036368,0.471278,-0.154151,0.174622,-0.468861,-0.035545,0.104261,0.005666,-0.044150
404,-0.415165,-0.176130,-0.131678,-0.180353,0.251314,0.862224,0.194156,-0.149489,0.163420,0.036368,-0.034804,0.264053,0.174622,0.214468,-0.035545,0.039556,-0.029853,-0.044150
940,1.184134,0.251314,-0.082842,-0.180353,2.043074,0.862224,-0.579034,0.211784,0.163420,-0.010050,0.051449,0.264053,0.174622,0.214468,-0.035545,-0.273497,-0.029853,0.066521
832,-0.763916,-0.474623,-1.090920,-0.241162,-0.965081,-0.301787,0.194156,0.211784,0.163420,0.036368,-0.034804,-0.117337,0.174622,-0.468861,0.060961,0.039556,0.005666,-0.044150
575,1.184134,0.251314,0.699082,-0.111591,0.251314,-0.301787,0.517018,0.211784,-0.246088,-0.010050,-0.034804,-0.587015,-0.746202,0.214468,0.060961,0.039556,0.005666,-0.044150
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,-0.763916,-0.987060,-1.090920,-0.111591,-0.311780,-0.301787,0.074691,-0.100083,0.163420,0.036368,-0.684779,0.264053,0.174622,-0.497095,0.060961,0.039556,-0.029853,-0.044150
132,1.184134,0.251314,-0.082842,-0.111591,0.239754,-0.301787,-0.579034,0.211784,0.163420,-0.010050,0.051449,-0.117337,-0.746202,0.214468,0.060961,0.104261,0.005666,-0.044150
679,1.184134,-0.176130,-0.082842,0.384846,0.239754,-0.301787,-0.579034,-0.100083,-0.246088,-0.071459,0.471278,-0.117337,0.174622,0.214468,-0.035545,0.039556,0.005666,-0.044150
900,-0.763916,0.251314,0.699082,-0.180353,0.239754,-0.301787,0.194156,0.211784,0.163420,0.036368,0.051449,0.264053,-0.746202,-0.468861,-0.035545,0.039556,0.005666,0.066521


In [ ]:

df_woe_train.drop(columns=["other_debtors","foreign_worker"])

KeyError: "['other_debtors', 'foreign_worker'] not found in axis"

In [ ]:
y_train = pd.read_parquet("../data/processed/y_train.parquet").squeeze()


In [108]:
import parquet


df_woe_train.to_parquet("../data/processed/X_train_woe.parquet")
df_woe_test.to_parquet("../data/processed/X_test_woe.parquet")
df_woe_val.to_parquet("../data/processed/X_val_woe.parquet")


y_train.to_frame().to_parquet("../data/processed/y_train.parquet")
y_test.to_frame().to_parquet("../data/processed/y_test.parquet")
y_val.to_frame().to_parquet("../data/processed/y_val.parquet")

In [101]:
df_woe_test

,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status_sex,residence_since,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone
258,1.184134,0.251314,-0.082842,0.554501,2.043074,-0.098061,-0.579034,0.292136,-0.246088,0.036368,-0.034804,-0.587015,0.174622,0.214468,-0.035545,0.039556,0.005666,0.066521
662,1.184134,-0.176130,-0.082842,-0.111591,0.239754,-0.301787,0.194156,-0.149489,0.163420,-0.010050,0.471278,0.264053,0.174622,0.214468,0.060961,0.039556,0.005666,-0.044150
915,-0.415165,-0.987060,-1.090920,-0.111591,-0.965081,-0.301787,0.074691,0.292136,-0.246088,-0.010050,0.051449,-0.020619,-0.746202,0.214468,-0.035545,-0.273497,0.005666,0.066521
192,-0.415165,-0.176130,-0.082842,-0.241162,-0.311780,-0.301787,0.074691,-0.149489,0.163420,-0.010050,-0.034804,0.264053,0.174622,0.214468,-0.035545,0.039556,-0.029853,0.066521
570,-0.763916,-0.176130,-0.082842,-0.111591,0.251314,-0.301787,-0.579034,-0.149489,-0.246088,0.036368,0.471278,-0.587015,0.174622,-0.468861,-0.035545,0.104261,0.005666,0.066521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533,1.184134,-0.176130,-0.082842,0.384846,0.128712,-0.098061,0.517018,-0.149489,0.079464,-0.010050,0.051449,-0.117337,0.174622,0.214468,-0.035545,0.039556,0.005666,0.066521
696,-0.415165,0.251314,-0.082842,0.384846,0.128712,-0.301787,0.517018,-0.149489,0.163420,-0.010050,0.471278,-0.117337,0.174622,0.214468,0.060961,0.039556,0.005666,-0.044150
993,-0.763916,-0.474623,-0.082842,-0.111591,-0.311780,-0.301787,-0.684779,-0.149489,0.163420,-0.010050,0.051449,-0.020619,0.174622,0.214468,-0.035545,-0.273497,0.005666,0.066521
47,-0.763916,1.455287,-0.082842,0.554501,0.128712,0.733153,-0.684779,0.292136,-0.246088,-0.010050,0.051449,-0.587015,0.174622,-0.468861,-0.035545,-0.273497,0.005666,0.066521


In [88]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

def check_woe_data(X_woe):
    """Vérifie la qualité des données WoE avant modélisation."""
    # Manquants
    missing = X_woe.isna().sum()
    print("Manquants par variable :\n", missing[missing > 0])
    
    # Types
    print("Dtypes :\n", X_woe.dtypes.value_counts())
    
    # VIF
    vif_df = pd.DataFrame({
        "feature": X_woe.columns,
        "VIF": [variance_inflation_factor(X_woe.values, i) for i in range(X_woe.shape[1])]
    }).sort_values("VIF", ascending=False)
    print(vif_df)
    return vif_df

vif_df = check_woe_data(df_woe_train)

Manquants par variable :
 Series([], dtype: int64)
Dtypes :
 float64    20
Name: count, dtype: int64
                    feature       VIF
11                 property  1.420950
14                  housing  1.389527
2            credit_history  1.285674
16                      job  1.267162
18                telephone  1.247017
15         existing_credits  1.193840
8       personal_status_sex  1.185561
10          residence_since  1.184039
12                      age  1.179976
1                  duration  1.173917
0           checking_status  1.153632
6                employment  1.152287
17           num_dependents  1.132850
5                   savings  1.097013
13  other_installment_plans  1.095452
7          installment_rate  1.069152
4             credit_amount  1.065643
3                   purpose  1.045249
9             other_debtors  1.000000
19           foreign_worker  1.000000


/tmp/ipykernel_21286/596163900.py:17: UserWarning: The design matrix is poorly conditioned (condition number=5.14e+31). VIF calculations may be numerically unstable.
  "VIF": [variance_inflation_factor(X_woe.values, i) for i in range(X_woe.shape[1])]
/home/osiris/WORK/PROJETS_STAGES/SCORECARD_CREDIT/.venv/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/osiris/WORK/PROJETS_STAGES/SCORECARD_CREDIT/.venv/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/osiris/WORK/PROJETS_STAGES/SCORECARD_CREDIT/.venv/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficien